# Forecasting del mercato del lavoro italiano

Previsione del tasso di disoccupazione/occupazione per regione a partire dal layer **gold** del warehouse (popolato dal DAG Airflow `mercato_lavoro_istat`).

Metodo: SARIMAX (statsmodels) con backtest sugli ultimi periodi disponibili, valutato con MAPE/RMSE.

**Prerequisito**: il DAG deve aver completato almeno un run con successo (`gold.fact_mercato_lavoro` non vuota).

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.metrics import mean_absolute_percentage_error, root_mean_squared_error
from sqlalchemy import create_engine
from statsmodels.tsa.statespace.sarimax import SARIMAX

from src.config import warehouse_connection_string

engine = create_engine(warehouse_connection_string())

## Caricamento dati dal layer gold

In [ ]:
fact = pd.read_sql('SELECT * FROM gold.fact_mercato_lavoro', engine)
dim_territorio = pd.read_sql('SELECT * FROM gold.dim_territorio', engine)
dim_tempo = pd.read_sql('SELECT * FROM gold.dim_tempo', engine)

df = fact.merge(dim_territorio, on='cod_territorio').merge(dim_tempo, on='periodo')
df.head()

## Selezione serie storica

Cambia `INDICATORE` e `TERRITORIO` per esplorare altre combinazioni (usa `df['indicatore'].unique()` / `df['territorio'].unique()` per vedere le opzioni disponibili).

In [ ]:
INDICATORE = 'tasso_disoccupazione'
TERRITORIO = 'Italia'  # sostituisci con una regione presente nei tuoi dati, es. 'Lombardia'
SESSO = 'totale'
CLASSE_ETA = 'totale'

serie = (
    df[(df['indicatore'] == INDICATORE) & (df['territorio'] == TERRITORIO)
       & (df['sesso'] == SESSO) & (df['classe_eta'] == CLASSE_ETA)]
    .sort_values('periodo')
    .set_index('periodo')['valore']
)
serie.index = pd.PeriodIndex(serie.index, freq='Q') if serie.index.str.contains('Q').all() else pd.to_datetime(serie.index)
serie = serie.sort_index()
px.line(serie, title=f'{INDICATORE} - {TERRITORIO}')

## Backtest: train/test split sugli ultimi periodi

In [ ]:
N_TEST = 4  # ultimi 4 periodi tenuti come test set
train, test = serie.iloc[:-N_TEST], serie.iloc[-N_TEST:]

model = SARIMAX(train, order=(1, 1, 1), seasonal_order=(0, 1, 1, 4), enforce_stationarity=False)
fit = model.fit(disp=False)

forecast_test = fit.get_forecast(steps=N_TEST).predicted_mean
forecast_test.index = test.index

mape = mean_absolute_percentage_error(test, forecast_test)
rmse = root_mean_squared_error(test, forecast_test)
print(f'MAPE: {mape:.2%}  |  RMSE: {rmse:.2f} punti percentuali')

In [ ]:
confronto = pd.concat([
    train.rename('storico'),
    test.rename('reale (test)'),
    forecast_test.rename('previsto (test)'),
], axis=1)
px.line(confronto, title=f'Backtest {INDICATORE} - {TERRITORIO} (MAPE {mape:.1%})')

## Previsione sui prossimi periodi (rifit su tutta la storia)

In [ ]:
N_FUTURE = 8
final_model = SARIMAX(serie, order=(1, 1, 1), seasonal_order=(0, 1, 1, 4), enforce_stationarity=False)
final_fit = final_model.fit(disp=False)
future_forecast = final_fit.get_forecast(steps=N_FUTURE).predicted_mean

storico_e_previsione = pd.concat([serie.rename('storico'), future_forecast.rename('previsione')])
px.line(storico_e_previsione, title=f'{INDICATORE} - {TERRITORIO}: storico + previsione a {N_FUTURE} periodi')

## (Opzionale) Scrivere le previsioni in `gold.forecast`

Così la dashboard Streamlit può mostrarle senza rieseguire il notebook. Ripeti il ciclo per tutte le combinazioni indicatore/territorio che ti interessano.

In [ ]:
risultati = future_forecast.reset_index()
risultati.columns = ['periodo', 'valore_previsto']
risultati['periodo'] = risultati['periodo'].astype(str)
risultati['indicatore'] = INDICATORE
risultati['territorio'] = TERRITORIO

# risultati.to_sql('forecast', engine, schema='gold', if_exists='append', index=False)
risultati